In [119]:
# 1. Construct a labeled DataFrame while preserving original SECOM record identifiers.
# 2. Create a reproducible balanced sample containing min(20+ ID_SUM, Pass count, Fail count) records per outcome.
# 3. Select five usable measurement features and apply median imputation using medians calculated from the full dataset.
# 4. Use .groupby() and .agg() to calculate counts, means, medians, and sample standard deviations for both outcome groups.
# 5. Calculate the absolute mean difference and standardized mean difference using the pooled withingroup sample standard deviation. Handle zero denominators safely.
# 6. Rank the selected features by absolute standardized mean difference and export the comparison.



In [120]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd
LAST_NAME = "Celo"
STUDENT_ID = "TUPM-26-1941"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT
np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


Student: Celo TUPM-26-1941
Seed Digit: 1
ID Sum: 23
Name Length: 4
Random Seed: 261941
Verification Value: 93


In [121]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\admin\Lab7_NumPy_Pandas_Celo
All dataset files verified.


In [122]:
# Load the SECOM measurement matrix and identify features containing missing or constant observed values.
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))


In [123]:
# 1. Construct a labeled DataFrame while preserving original SECOM record identifiers.
rng = np.random.default_rng(RANDOM_SEED)

feature = raw_measurements.shape[1]
features = [f"Feature_{i:03d}" for i in range(feature)]

table = pd.DataFrame(raw_measurements.copy(), columns=features)
table.insert(0,"Record_ID", np.arange(len(table)))

labels = np.loadtxt(label_path, usecols=0, dtype=int)
table["Result"] = labels
table["Outcome"] = np.where(labels == -1, "Pass", "Fail")


In [124]:
# 2. Create a reproducible balanced sample containing min(20+ ID_SUM, Pass count, Fail count) records per outcome.
p_record = table[table["Outcome"] =="Pass"]
f_record = table[table["Outcome"] =="Fail"]

sample = min(20 + ID_SUM, len(p_record), len(f_record))

pass_s = rng.choice(len(p_record), size=sample, replace=False)
fail_s = rng.choice(len(f_record), size=sample, replace=False)

balanced = pd.concat(
    [p_record.iloc[pass_s], f_record.iloc[fail_s]],
    ignore_index=True
)


In [125]:
# 3. Select five usable measurement features and apply median imputation using medians calculated from the full dataset.
selected = rng.choice(usable_indices, size=5, replace=False)
feature_s = [features[i] for i in selected]

dataset = table[feature_s].median()

balanced = balanced[["Record_ID", "Result", "Outcome"] + feature_s].copy()
balanced[feature_s] = balanced[feature_s].fillna(dataset)


In [126]:
# 4. Use .groupby() and .agg() to calculate counts, means, medians, and sample standard deviations for both outcome groups.
grouped = balanced.groupby("Outcome")[feature_s].agg(
    ["count", "mean", "median", "std"]
)


In [127]:
# 5. Calculate the absolute mean difference and standardized mean difference using the pooled withingroup sample standard deviation. Handle zero denominators safely.
calculate = []
verify = True
for feat in feature_s:
    p_count = grouped.loc["Pass", (feat, "count")]
    p_mean = grouped.loc["Pass", (feat, "mean")]
    p_median = grouped.loc["Pass", (feat, "median")]
    p_std = grouped.loc["Pass", (feat, "std")]

    f_count = grouped.loc["Fail", (feat, "count")]
    f_mean = grouped.loc["Fail", (feat, "mean")]
    f_median = grouped.loc["Fail", (feat, "median")]
    f_std = grouped.loc["Fail", (feat, "std")]

    v_pass = balanced.loc[balanced["Outcome"] == "Pass", feat].to_numpy()
    v_fail = balanced.loc[balanced["Outcome"] == "Fail", feat].to_numpy()

    if not np.allclose(
        [p_mean, np.median(v_pass), p_std, f_mean, np.median(v_fail), f_std],
        [v_pass.mean(), np.median(v_pass), v_pass.std(ddof=1), v_fail.mean(), np.median(v_fail), v_fail.std(ddof=1)]
    ):
        verify = False

    a_mean = abs(f_mean - p_mean)

    dof = (p_count + f_count - 2)
    v_pooled = (((p_count - 1) * (p_std ** 2)) + ((f_count - 1) * (f_std ** 2))) / dof if dof > 0 else 0.0
    sd_pooled = np.sqrt(v_pooled)

    a_smd = a_mean / sd_pooled if sd_pooled > 0 else 0.0

    calculate.append({
        "Feature": feat,
        "Pass - Count": int(p_count),
        "Pass - Mean": p_mean,
        "Pass - Median": p_median,
        "Pass - Standard Deviation": p_std,
        "Fail - Count": int(f_count),
        "Fail - Mean": f_mean,
        "Fail - Median": f_median,
        "Fail - Standard Deviation": f_std,
        "Absolute Mean Difference": a_mean,
        "Pooled Standard Deviation": sd_pooled,
        "Absolute Standardized Mean Difference": a_smd
    })

In [128]:
# 6. Rank the selected features by absolute standardized mean difference and export the comparison.
compare = (
    pd.DataFrame(calculate)
    .sort_values(by="Absolute Standardized Mean Difference", ascending=False)
    .reset_index(drop=True)
)

balanced.to_csv("balanced_sample.csv", index=False)
compare.to_csv("feature_comparison.csv", index=False)


In [129]:
# [Assessment Data] Output Parameter
print("Balanced Sample Size:", len(balanced))
print("Selected Features:", ", ".join(feature_s))
print("Highest-Ranked Feature:", compare.loc[0, "Feature"])
print(f"Absolute Standardized Mean Difference: {compare.loc[0, 'Absolute Standardized Mean Difference']:.4f}")
print("NumPy-Pandas Statistics Match:", verify)

Balanced Sample Size: 86
Selected Features: Feature_356, Feature_044, Feature_252, Feature_296, Feature_182
Highest-Ranked Feature: Feature_296
Absolute Standardized Mean Difference: 0.2913
NumPy-Pandas Statistics Match: True
